In [1]:
# Ensure we're running from the project root (parent of notebooks/)
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print('Working dir:', Path.cwd())

Working dir: /mnt/d/AI/Projects/slonik-7b


# 01 — Baseline & Setup

Before any fine-tuning, where does the base Qwen2.5-Coder-7B-Instruct land on BIRD Mini-Dev PostgreSQL?

| Model | BIRD-PG | Size |
|---|---:|---|
| Qwen2.5-Coder-7B (base) | **12.22%** | 7B |

This notebook documents the baseline setup, the eval methodology, and the questions the project tries to answer.


## Project goal

Build a small text-to-SQL model that:
- Handles PostgreSQL-specific patterns (JSONB, pgvector, full-text search, window functions, CTEs)
- Runs locally on a laptop GPU (16 GB VRAM)
- Beats the closest open and commercial baselines on a real benchmark

The baseline says we have a long way to go: 12.22% on BIRD-PG means roughly 88% of queries fail or return wrong results.


## Evaluation setup

We use [BIRD Mini-Dev](https://bird-bench.github.io/), a 500-example dev set with PostgreSQL and SQLite versions of the same questions. The evaluation methodology is **execution accuracy**:

1. Render the schema as `CREATE TABLE` statements
2. Send schema + question (+ optional evidence hint) to the model
3. Extract the SQL from the model output
4. Execute the gold SQL and the predicted SQL against the real database
5. Compare result sets, order-independent for non-`ORDER BY` queries

This matches the official BIRD evaluation protocol.


In [2]:
# Connect to the Postgres instance loaded with the BIRD dump
# (See docker setup in scripts/eval_bird_pg.py and the GitHub README)

import psycopg
import json
from pathlib import Path

conn = psycopg.connect("host=localhost port=5432 user=postgres password=slonik dbname=postgres")

# Verify the BIRD tables landed
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM information_schema.tables WHERE table_schema='public'")
    print(f"Tables loaded in Postgres: {cur.fetchone()[0]}")

Tables loaded in Postgres: 75


## Load a sample question

Each entry has a `db_id`, `question`, gold `SQL`, optional `evidence`, and a `difficulty` tier (simple/moderate/challenging).


In [3]:
# Load BIRD Mini-Dev PG questions
questions_path = Path("data/raw/bird_mini_dev/mini_dev_pg.jsonl")
with open(questions_path) as f:
    questions = [json.loads(line) for line in f if line.strip()]

print(f"Total questions: {len(questions)}")
print()
print("Sample question:")
print(json.dumps(questions[0], indent=2)[:600])

Total questions: 500

Sample question:
{
  "question_id": 1471,
  "db_id": "debit_card_specializing",
  "question": "What is the ratio of customers who pay in EUR against customers who pay in CZK?",
  "evidence": "ratio of customers who pay in EUR against customers who pay in CZK = count(Currency = 'EUR') / count(Currency = 'CZK').",
  "SQL": "SELECT CAST(SUM(CASE WHEN Currency = 'EUR' THEN 1 ELSE 0 END) AS REAL) / NULLIF(SUM(CASE WHEN Currency = 'CZK' THEN 1 ELSE 0 END), 0) FROM customers",
  "difficulty": "simple"
}


## Difficulty distribution

In [4]:
from collections import Counter
diffs = Counter(q.get("difficulty", "?") for q in questions)
for d in ("simple", "moderate", "challenging"):
    print(f"  {d:13s}: {diffs[d]}")

  simple       : 148
  moderate     : 250
  challenging  : 102


## What the base model does

Before fine-tuning, the base Qwen2.5-Coder-7B-Instruct treats text-to-SQL like generic code generation. It produces syntactically valid SQL but:
- Often uses MySQL-style functions (`MONTH()`, `YEAR()`) that fail on Postgres
- Frequently hallucinates column names not present in the schema
- Misses PostgreSQL-specific patterns entirely (JSONB ops, `EXTRACT`, `ILIKE`, etc.)

The 12.22% score reflects this gap. The rest of the project addresses each of those failure modes.


## Roadmap of remaining notebooks

| Notebook | Topic | Result |
|---|---|---|
| `02_sft_training_results` | QLoRA SFT on 21K examples | 33.20% BIRD-PG |
| `03_grpo_results_and_comparison` | 2000-step GRPO with execution rewards | 38.20% BIRD-PG, 45.20% BIRD-SQLite |
| `04_error_analysis` | What's still failing and why | — |
